# **PostgreSQL persistence validation**

Este notebook importa e valida as funções de `src/bibliometric_rag_assistant/db_postgres.py`, originadas da etapa exploratória no notebook `04_postgres_connection_exploration.ipynb`.

Todos os dados são fictícios e locais: nenhuma API externa é consultada. Cada execução criada aqui recebe um `run_id` único com prefixo `validation_`, é registrada em `created_run_ids` e apagada na seção de limpeza.

Pré-requisito: o container do `compose.yaml` em execução (`docker compose up -d`).

## **Imports**

In [158]:
import json
from datetime import UTC, datetime
from uuid import uuid4

from psycopg.errors import UniqueViolation
from sqlalchemy import func, inspect, select, text
from sqlalchemy.exc import IntegrityError

from bibliometric_rag_assistant import db_postgres as db_pg


## **1. Conexão e esquema**

In [159]:
engine = db_pg.create_postgres_engine()
db_pg.create_tables(engine)

In [160]:
inspector = inspect(engine)

expected_tables = {
    "ingestion_runs",
    "raw_snapshots",
    "citations",
    "citation_enrichments",
}

assert expected_tables <= set(inspector.get_table_names())

inspector.get_table_names()

['ingestion_runs', 'raw_snapshots', 'citations', 'citation_enrichments']

`create_all()` só cria tabelas ausentes: não altera uma tabela que já existe. Se `raw_snapshots` tiver sido criada antes do ajuste de `file_path`, o assert abaixo falha e a tabela precisa ser recriada.

In [161]:
snapshot_columns = {
    column["name"]: column
    for column in inspector.get_columns("raw_snapshots")
}

assert snapshot_columns["file_path"]["nullable"] is True
assert snapshot_columns["payload"]["nullable"] is False

Buscas independentes não têm variante nem versão do preset: `query_variant` e `query_version` passaram a aceitar `NULL` em `db_postgres.py`. Como `create_all()` não altera colunas de tabelas existentes, a mudança foi aplicada ao banco com o SQL abaixo, que preserva os dados e pode ser executado de novo sem efeito.

In [ ]:
with engine.begin() as connection:
    connection.execute(text("""
        ALTER TABLE ingestion_runs
            ALTER COLUMN query_variant DROP NOT NULL,
            ALTER COLUMN query_version DROP NOT NULL
    """))

In [ ]:
# Inspector novo: o anterior guarda as colunas lidas antes do ALTER
run_columns = {
    column["name"]: column
    for column in inspect(engine).get_columns("ingestion_runs")
}

assert run_columns["query_variant"]["nullable"] is True
assert run_columns["query_version"]["nullable"] is True
assert run_columns["query_text"]["nullable"] is False

O banco pode já conter dados reais. As contagens antes de qualquer inserção servem de linha de base para as seções 6 e 7, que comparam contra ela e não contra zero.

In [162]:
with engine.connect() as connection:
    initial_counts = db_pg.count_rows(connection)

initial_counts

{'ingestion_runs': 2,
 'citations': 3,
 'raw_snapshots': 3,
 'citation_enrichments': 4}

## **Funções auxiliares do notebook**

`make_run()` monta uma execução fictícia e registra seu `run_id` para a limpeza. `count_run_rows()` conta as linhas de um `run_id` em cada tabela, na ordem das dependências (`metadata.sorted_tables`), sempre numa conexão nova.

In [163]:
created_run_ids = []


def make_run(label: str, total: int) -> dict:
    run_id = f"validation_{label}_{uuid4().hex}"
    created_run_ids.append(run_id)

    return {
        "run_id": run_id,
        "source": "validation",
        "query_text": "Validação da persistência no PostgreSQL",
        "query_variant": "persistence_validation",
        "query_version": "0.1",
        "retrieved_at": datetime.now(UTC).isoformat(),
        "total_results": total,
        "total_retrieved": total,
    }


def count_run_rows(run_id: str) -> dict[str, int]:
    with engine.connect() as connection:
        return {
            table.name: connection.execute(
                select(func.count())
                .select_from(table)
                .where(table.c.run_id == run_id)
            ).scalar_one()
            for table in db_pg.metadata.sorted_tables
        }

## **2. Ingestão controlada**

Uma execução com dois snapshots (ESearch em JSON, EFetch em XML) e três citações. As citações cobrem: autores e afiliações com acentos, DOI ausente e chaves `authors`/`affiliations` ausentes, que devem virar listas vazias.

In [164]:
run = make_run("ingestion", total=3)
run_id = run["run_id"]

esearch_payload = {
    "esearchresult": {
        "count": "3",
        "idlist": ["VAL001", "VAL002", "VAL003"],
    }
}

efetch_payload = (
    "<PubmedArticleSet>"
    "<PubmedArticle><PMID>VAL001</PMID></PubmedArticle>"
    "<PubmedArticle><PMID>VAL002</PMID></PubmedArticle>"
    "<PubmedArticle><PMID>VAL003</PMID></PubmedArticle>"
    "</PubmedArticleSet>"
)

snapshots = [
    {
        "run_id": run_id,
        "endpoint": "esearch",
        "payload_format": "json",
        "payload": esearch_payload,
    },
    {
        "run_id": run_id,
        "endpoint": "efetch",
        "payload_format": "xml",
        "payload": efetch_payload,
    },
]

In [165]:
records = [
    {
        "pmid": "VAL001",
        "doi": "10.0000/val.001",
        "title": "Artigo fictício com DOI",
        "abstract": "Resumo fictício.",
        "journal": "Periódico Fictício",
        "publication_year": 2026,
        "language": "por",
        "authors": ["Ana Souza", "João Lima"],
        "affiliations": ["Universidade Fictícia, Niterói"],
    },
    {
        "pmid": "VAL002",
        "doi": None,
        "title": "Artigo fictício sem DOI",
        "publication_year": 2025,
        "language": "eng",
        "authors": ["Maria Pereira"],
        "affiliations": [],
    },
    {
        "pmid": "VAL003",
        "doi": "10.0000/val.003",
        "title": "Artigo fictício sem autores e afiliações",
    },
]

In [166]:
db_pg.save_ingestion(engine, run, snapshots, records)

In [167]:
assert count_run_rows(run_id) == {
    "ingestion_runs": 1,
    "raw_snapshots": 2,
    "citations": 3,
    "citation_enrichments": 0,
}

In [168]:
run_id

'validation_ingestion_513049c1ef354012b814477af7d82dea'

In [169]:
with engine.connect() as connection:
    saved_run = connection.execute(
        select(db_pg.ingestion_runs)
        .where(db_pg.ingestion_runs.c.run_id == run_id)
    ).mappings().one()

    saved_snapshots = connection.execute(
        select(db_pg.raw_snapshots)
        .where(db_pg.raw_snapshots.c.run_id == run_id)
    ).mappings().all()

    saved_citations = connection.execute(
        select(db_pg.citations)
        .where(db_pg.citations.c.run_id == run_id)
        .order_by(db_pg.citations.c.pmid)
    ).mappings().all()

assert dict(saved_run) == run

O payload JSON foi serializado por `insert_raw_snapshot()` e volta igual após `json.loads()`. O XML já era texto e é gravado como está. Sem `file_path` nos snapshots, a coluna fica `NULL`.

In [170]:
snapshots_by_endpoint = {row["endpoint"]: row for row in saved_snapshots}

esearch_row = snapshots_by_endpoint["esearch"]
efetch_row = snapshots_by_endpoint["efetch"]

assert esearch_row["payload_format"] == "json"
assert json.loads(esearch_row["payload"]) == esearch_payload

assert efetch_row["payload_format"] == "xml"
assert efetch_row["payload"] == efetch_payload

assert all(row["file_path"] is None for row in saved_snapshots)

In [171]:
authors = {
    row["pmid"]: json.loads(row["authors_json"])
    for row in saved_citations
}
affiliations = {
    row["pmid"]: json.loads(row["affiliations_json"])
    for row in saved_citations
}

assert authors == {
    "VAL001": ["Ana Souza", "João Lima"],
    "VAL002": ["Maria Pereira"],
    "VAL003": [],
}
assert affiliations == {
    "VAL001": ["Universidade Fictícia, Niterói"],
    "VAL002": [],
    "VAL003": [],
}

# ensure_ascii=False: acentos gravados como texto, não como \u escapes
assert "João" in saved_citations[0]["authors_json"]

In [172]:
first, second, third = saved_citations

assert first["doi"] == "10.0000/val.001"
assert first["publication_year"] == 2026
assert first["journal"] == "Periódico Fictício"

assert second["doi"] is None
assert second["abstract"] is None

assert third["journal"] is None
assert third["publication_year"] is None

## **3. Enriquecimento controlado**

Um caso de cada status, sobre as citações da execução anterior (a FK exige que `(run_id, pmid)` exista em `citations`):

- `success`: `VAL001`, com resposta bruta e registro normalizado;
- `missing_doi`: `VAL002`, sem DOI e sem dados;
- `not_found`: `VAL003`, com DOI mas sem resposta (404).

In [173]:
retrieved_at = datetime.now(UTC).isoformat()

crossref_raw = {
    "status": "ok",
    "message": {
        "DOI": "10.0000/val.001",
        "title": ["Artigo fictício com DOI"],
        "author": [
            {"given": "Ana", "family": "Souza"},
            {"given": "João", "family": "Lima"},
        ],
    },
}

crossref_normalized = {
    "doi": "10.0000/val.001",
    "title": "Artigo fictício com DOI",
    "authors": ["Ana Souza", "João Lima"],
}

In [174]:
enrichments = [
    {
        "run_id": run_id,
        "pmid": "VAL001",
        "source": "crossref",
        "doi": "10.0000/val.001",
        "retrieved_at": retrieved_at,
        "status": "success",
        "raw": crossref_raw,
        "normalized": crossref_normalized,
    },
    {
        "run_id": run_id,
        "pmid": "VAL002",
        "source": "crossref",
        "doi": None,
        "retrieved_at": retrieved_at,
        "status": "missing_doi",
    },
    {
        "run_id": run_id,
        "pmid": "VAL003",
        "source": "crossref",
        "doi": "10.0000/val.003",
        "retrieved_at": retrieved_at,
        "status": "not_found",
        "raw": None,
        "normalized": None,
    },
]

db_pg.save_citation_enrichments(engine, enrichments)

In [175]:
with engine.connect() as connection:
    saved_enrichments = connection.execute(
        select(db_pg.citation_enrichments)
        .where(db_pg.citation_enrichments.c.run_id == run_id)
    ).mappings().all()

enrichments_by_status = {row["status"]: row for row in saved_enrichments}

assert set(enrichments_by_status) == {"success", "missing_doi", "not_found"}
assert count_run_rows(run_id)["citation_enrichments"] == 3

In [176]:
success = enrichments_by_status["success"]

assert success["pmid"] == "VAL001"
assert json.loads(success["raw_json"]) == crossref_raw
assert json.loads(success["normalized_json"]) == crossref_normalized

`None` no Python poderia vir tanto de SQL `NULL` quanto ser confundido com a string JSON `"null"`. O filtro `IS NULL` no próprio banco confirma que `dumps_or_none()` gravou `NULL` de fato.

In [177]:
enrichments_table = db_pg.citation_enrichments

with engine.connect() as connection:
    null_statuses = connection.execute(
        select(enrichments_table.c.status)
        .where(enrichments_table.c.run_id == run_id)
        .where(enrichments_table.c.raw_json.is_(None))
        .where(enrichments_table.c.normalized_json.is_(None))
    ).scalars().all()

assert set(null_statuses) == {"missing_doi", "not_found"}
assert enrichments_by_status["missing_doi"]["doi"] is None
assert enrichments_by_status["not_found"]["doi"] == "10.0000/val.003"

## **4. Listas vazias**

Uma busca sem resultados ainda gera a execução e o snapshot do ESearch, mas nenhuma citação. As funções de inserção pulam listas vazias, então ambas as chamadas devem terminar sem erro e sem linhas extras.

In [178]:
empty_run = make_run("empty", total=0)
empty_run_id = empty_run["run_id"]

empty_snapshots = [
    {
        "run_id": empty_run_id,
        "endpoint": "esearch",
        "payload_format": "json",
        "payload": {"esearchresult": {"count": "0", "idlist": []}},
    }
]

db_pg.save_ingestion(engine, empty_run, empty_snapshots, records=[])

assert count_run_rows(empty_run_id) == {
    "ingestion_runs": 1,
    "raw_snapshots": 1,
    "citations": 0,
    "citation_enrichments": 0,
}

In [179]:
def count_enrichments() -> int:
    with engine.connect() as connection:
        return connection.execute(
            select(func.count()).select_from(db_pg.citation_enrichments)
        ).scalar_one()


enrichments_before = count_enrichments()

db_pg.save_citation_enrichments(engine, enrichments=[])

assert count_enrichments() == enrichments_before

## **5. Rollback**

Duas citações com o mesmo PMID violam a chave primária `(run_id, pmid)`. Como a execução e o snapshot são inseridos antes das citações, o teste só passa se `engine.begin()` desfizer a transação inteira. O `IntegrityError` é capturado aqui, fora de `save_ingestion()`.

In [180]:
broken_run = make_run("rollback", total=2)
broken_run_id = broken_run["run_id"]

broken_snapshots = [
    {
        "run_id": broken_run_id,
        "endpoint": "esearch",
        "payload_format": "json",
        "payload": {"esearchresult": {"count": "2", "idlist": ["VAL900"]}},
    }
]

duplicated_records = [
    {"pmid": "VAL900", "title": "Primeira ocorrência"},
    {"pmid": "VAL900", "title": "PMID duplicado"},
]

In [181]:
try:
    db_pg.save_ingestion(engine, broken_run, broken_snapshots, duplicated_records)
except IntegrityError as error:
    rollback_error = error
else:
    raise AssertionError("O PMID duplicado deveria ter gerado IntegrityError")

assert isinstance(rollback_error.orig, UniqueViolation)

type(rollback_error.orig).__name__

'UniqueViolation'

In [182]:
assert count_run_rows(broken_run_id) == {
    "ingestion_runs": 0,
    "raw_snapshots": 0,
    "citations": 0,
    "citation_enrichments": 0,
}

## **6. Consultas**

As consultas usam os dados das seções anteriores. Cada célula abre sua própria conexão com `engine.connect()`, como fará o chamador.

Uma segunda fonte (`openalex`) processa apenas `VAL001`. Ela permite conferir o filtro por fonte e a diferença entre ausências registradas e citações ainda não processadas.

In [183]:
db_pg.save_citation_enrichments(
    engine,
    [
        {
            "run_id": run_id,
            "pmid": "VAL001",
            "source": "openalex",
            "doi": "10.0000/val.001",
            "retrieved_at": datetime.now(UTC).isoformat(),
            "status": "success",
            "raw": {"id": "W0000000001"},
            "normalized": {"doi": "10.0000/val.001"},
        }
    ],
)

### **`get_latest_run_id()`**

A execução vazia foi a última salva: a do rollback não persistiu. Os `retrieved_at` são ISO 8601 em UTC, então a ordem textual coincide com a cronológica.

O retorno `None` só ocorre com o banco vazio e não é exercitado aqui, para não tocar nos dados reais.

In [184]:
with engine.connect() as connection:
    latest_run_id = db_pg.get_latest_run_id(connection)

assert latest_run_id == empty_run_id

### **`load_citations()`**

As citações voltam com o mesmo formato dos `records` de entrada, acrescidas de `run_id`: `authors_json` e `affiliations_json` são desserializados em `authors` e `affiliations`, e campos ausentes na entrada voltam como `None` ou `[]`.

In [185]:
with engine.connect() as connection:
    loaded_citations = db_pg.load_citations(connection, run_id)
    empty_run_citations = db_pg.load_citations(connection, empty_run_id)
    missing_run_citations = db_pg.load_citations(connection, "validation_missing_run")

citation_defaults = {
    "doi": None,
    "title": None,
    "abstract": None,
    "journal": None,
    "publication_year": None,
    "language": None,
    "authors": [],
    "affiliations": [],
}

assert loaded_citations == [
    {"run_id": run_id, **citation_defaults, **record}
    for record in records
]
assert "authors_json" not in loaded_citations[0]

assert empty_run_citations == []
assert missing_run_citations == []

### **`get_enriched_pmids()`**

In [186]:
with engine.connect() as connection:
    crossref_pmids = db_pg.get_enriched_pmids(connection, run_id, "crossref")
    openalex_pmids = db_pg.get_enriched_pmids(connection, run_id, "openalex")
    unknown_source_pmids = db_pg.get_enriched_pmids(connection, run_id, "unknown_source")
    empty_run_pmids = db_pg.get_enriched_pmids(connection, empty_run_id, "crossref")

# success, missing_doi e not_found contam como processados
assert crossref_pmids == {"VAL001", "VAL002", "VAL003"}
assert openalex_pmids == {"VAL001"}

assert unknown_source_pmids == set()
assert empty_run_pmids == set()

As citações ainda não processadas por uma fonte são a diferença entre as citações da execução e os PMIDs processados. É o que a retomada vai consultar.

In [187]:
run_pmids = {citation["pmid"] for citation in loaded_citations}

assert run_pmids - crossref_pmids == set()
assert run_pmids - openalex_pmids == {"VAL002", "VAL003"}

### **`load_enrichment_gaps()`**

Só as ausências já registradas entram: no `crossref`, `VAL002` (`missing_doi`) e `VAL003` (`not_found`). No `openalex`, essas citações não foram processadas, então não são lacunas.

In [188]:
with engine.connect() as connection:
    crossref_gaps = db_pg.load_enrichment_gaps(connection, run_id, "crossref")
    openalex_gaps = db_pg.load_enrichment_gaps(connection, run_id, "openalex")
    empty_run_gaps = db_pg.load_enrichment_gaps(connection, empty_run_id, "crossref")

assert crossref_gaps == [
    {
        "status": "missing_doi",
        "pmid": "VAL002",
        "doi": None,
        "title": "Artigo fictício sem DOI",
    },
    {
        "status": "not_found",
        "pmid": "VAL003",
        "doi": "10.0000/val.003",
        "title": "Artigo fictício sem autores e afiliações",
    },
]

assert openalex_gaps == []
assert empty_run_gaps == []

### **`load_raw_snapshot()`**

O payload JSON volta como objeto Python e o XML como o texto gravado. A busca vazia não chega ao EFetch, então esse snapshot não existe.

In [189]:
with engine.connect() as connection:
    loaded_esearch = db_pg.load_raw_snapshot(connection, run_id, "esearch")
    loaded_efetch = db_pg.load_raw_snapshot(connection, run_id, "efetch")
    missing_efetch = db_pg.load_raw_snapshot(connection, empty_run_id, "efetch")
    missing_run = db_pg.load_raw_snapshot(connection, "validation_missing_run", "esearch")

assert loaded_esearch["payload_format"] == "json"
assert loaded_esearch["payload"] == esearch_payload

assert loaded_efetch["payload_format"] == "xml"
assert loaded_efetch["payload"] == efetch_payload

assert loaded_esearch["file_path"] is None
assert loaded_efetch["file_path"] is None

assert missing_efetch is None
assert missing_run is None

### **`count_rows()`**

Os totais incluem os dados reais. A diferença em relação à linha de base deve ser exatamente o que o notebook inseriu: as execuções `ingestion` e `empty`, e quatro enriquecimentos (três do `crossref`, um do `openalex`).

In [190]:
with engine.connect() as connection:
    current_counts = db_pg.count_rows(connection)

assert set(current_counts) == expected_tables

assert {
    table: current_counts[table] - initial_counts[table]
    for table in current_counts
} == {
    "ingestion_runs": 2,
    "raw_snapshots": 3,
    "citations": 3,
    "citation_enrichments": 4,
}

current_counts

{'ingestion_runs': 4,
 'citations': 6,
 'raw_snapshots': 6,
 'citation_enrichments': 8}

## **7. Limpeza**

Apaga apenas os `run_id` criados por este notebook, das tabelas dependentes para as referenciadas (`reversed(metadata.sorted_tables)`): enriquecimentos, citações, snapshots e, por fim, as execuções.

In [191]:
with engine.begin() as connection:
    for table in reversed(db_pg.metadata.sorted_tables):
        connection.execute(
            table.delete().where(table.c.run_id.in_(created_run_ids))
        )

for created_run_id in created_run_ids:
    assert sum(count_run_rows(created_run_id).values()) == 0

with engine.connect() as connection:
    assert db_pg.count_rows(connection) == initial_counts

created_run_ids

['validation_ingestion_513049c1ef354012b814477af7d82dea',
 'validation_empty_063506f033454d1c9068aecc35c1fbde',
 'validation_rollback_5eac92aca5f346bc85143746506d3441']

In [192]:
engine.dispose()